Set the working directory properly 

In [11]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

Load the data

In [2]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("fancyzhx/yelp_polarity")

df_train = dataset["train"].to_pandas()
df_test = dataset["test"].to_pandas()

Apply the preprocessing

In [3]:
from src.preprocessing import clean_text

example = "   This   restaurant was AMAZING!!!   "

clean_text(example)

df_train_clean = df_train.copy()
df_test_clean = df_test.copy()

df_train_clean["text"] = df_train_clean["text"].apply(clean_text)
df_test_clean["text"] = df_test_clean["text"].apply(clean_text)

df_train_clean.head()

print(
    "Empty training texts:",
    (df_train_clean["text"] == "").sum()
)

print(
    "Empty test texts:",
    (df_test_clean["text"] == "").sum()
)

#There's a slight difference here compared to the EDA: no adding `text_length`. That was a temporary feature used for analysis.
print("Train:", df_train_clean.shape)
print("Test :", df_test_clean.shape)

Empty training texts: 0
Empty test texts: 0
Train: (560000, 2)
Test : (38000, 2)


Prepar X and y

In [4]:
X = df_train_clean["text"]
y = df_train_clean["label"]

X_test = df_test_clean["text"]
y_test = df_test_clean["label"]

'''
X       → recensioni
y       → sentiment
X_test  → recensioni test
y_test  → sentiment test
'''

'\nX       → recensioni\ny       → sentiment\nX_test  → recensioni test\ny_test  → sentiment test\n'

Create validation set

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Verify the sizes

In [6]:
print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)

print("y_train:", y_train.shape)
print("y_val  :", y_val.shape)
print("y_test :", y_test.shape)

X_train: (448000,)
X_val  : (112000,)
X_test : (38000,)
y_train: (448000,)
y_val  : (112000,)
y_test : (38000,)


Verify the balance (to be sure that the split didn't affect the distribution)

In [7]:
print("Training distribution:")
print(y_train.value_counts(normalize=True))

print("\nValidation distribution:")
print(y_val.value_counts(normalize=True))

print("\nTest distribution:")
print(y_test.value_counts(normalize=True))

Training distribution:
label
0    0.5
1    0.5
Name: proportion, dtype: float64

Validation distribution:
label
1    0.5
0    0.5
Name: proportion, dtype: float64

Test distribution:
label
1    0.5
0    0.5
Name: proportion, dtype: float64


Check to make sure there's no leakage.

If the same review were to appear in both the training and test sets, the model might have already “seen” it during training.

This would make the evaluation less reliable.

In [8]:
train_texts = set(X_train)
val_texts = set(X_val)
test_texts = set(X_test)

train_val_overlap = train_texts & val_texts
train_test_overlap = train_texts & test_texts
val_test_overlap = val_texts & test_texts

print("Train ∩ Validation:", len(train_val_overlap))
print("Train ∩ Test:", len(train_test_overlap))
print("Validation ∩ Test:", len(val_test_overlap))

Train ∩ Validation: 2
Train ∩ Test: 2
Validation ∩ Test: 0


This situation has to be verified:
- Duplicates in the original training didn't exists (0)
- Duplicates train cross Validation (2)
- So duplicates after preprocessing in the train (at least 2)

Following step is to check the texts involved in the cross sets.

In [9]:

print("\nTRAIN / VALIDATION")
for text in train_val_overlap:
    print(text)
    print("-" * 80)

print("\nTRAIN / TEST")
for text in train_test_overlap:
    print(text)
    print("-" * 80)


TRAIN / VALIDATION
Unbelievable!!! We order my mom, which is dying of cancer some \""edible arrangements\"" for Valentines day, hoping it would bring a smile to her face! All day long we tracked the order and it said \""in transit\"" the day came and went and my moms arrangement never showed up!!! Almost $60 bucks down the drain! We told my mom something was coming the day before, so she was excited, only nothing ever came!! And on top of it all, NOT EVEN A PHONE CALL letting us know that they were not going to be able to fulfill the order, at least if they had let us know we could have gotten her some flowers or something to replace it. THANKS \""EDIBLE ARRANGEMENTS\"", thanks a lot for let my mom down!!!!!!!! I'm not even mad about it, In fact I'm very sadden by it......
--------------------------------------------------------------------------------
BUYER AND CONSUMER BEWARE!!!! I called Gabriels to check out a 1 year old refrigerator that was working but had stopped cooling. The t

So we can suppose that the `clean_text()` may have caused the 2 different texts:
- "Great food!   Great service!"
- "Great food! Great service!"

to becoame the same:
- "Great food! Great service!"
- "Great food! Great service!"

Following step is to check:
1. the number of duplicates inside the training after the cleaning
2. the size of the split
3. any index overlap (if `train_test_split` assigned each row to an unique split)
4. the duplicated texts inside the training after the cleaning

In [10]:
print(
    "Duplicates after cleaning:",
    df_train_clean["text"].duplicated().sum()
)

print("Train rows:", len(X_train))
print("Validation rows:", len(X_val))

print("Index overlap:", len(set(X_train.index) & set(X_val.index)))

duplicates = df_train_clean[
    df_train_clean["text"].duplicated(keep=False)
].sort_values("text")

duplicates[["text", "label"]]

Duplicates after cleaning: 12
Train rows: 448000
Validation rows: 112000
Index overlap: 0


,text,label
439725,Amazeballs. Went for dessert (baked chocolate ...,1
439728,Amazeballs. Went for dessert (baked chocolate ...,1
208825,BUYER AND CONSUMER BEWARE!!!! I called Gabriel...,0
208826,BUYER AND CONSUMER BEWARE!!!! I called Gabriel...,0
452385,Bagel Man's rudeness exceeds his lack of busin...,0
452387,Bagel Man's rudeness exceeds his lack of busin...,0
85552,DO NOT GO HERE or believe their prices on the ...,0
85553,DO NOT GO HERE or believe their prices on the ...,0
190337,"Great food, I have tried many things on the me...",1
190340,"Great food, I have tried many things on the me...",1


1. The couples have **different index** and **same text**
  ↓
- Duplicate reviews are almost certainly duplicates of the original dataset in terms of content, but differ in some *whitespace characters*.

2. The couples have **different index** and **same text** and **same label**
  ↓
- Perfect! It is important to avoid any potential compromise of the sentiment detection


# Conclusion

- *Duplicates inside the training*: Have we to eliminate them?
No, due to they are only 12 collisions on 560000 => 0.002%

- *Duplicates cross split*: Have we to eliminate them?
(They are "Stayed 5 nights 10/29-11/2..." and "Great food! Great service!". They are not explained with the 12 collisions of the training because the test is separated)
No, due to ther are only 2 collisions on 38000 => 0.005%

- The initial check is not enoguh:
`df_train["text"].duplicated()`

- The correct check is: 
raw duplicates (0)
       ↓
preprocessed duplicates (12)
       ↓
cross-split duplicates (2)


**Here a detail of the checks' results:**

RAW DATA
   │
   ▼
clean_text()
   │
   ▼
12 duplicate collisions
   │
   ├── same label
   └── no conflicts
   │
   ▼
TRAIN / VALIDATION
   │
   ▼
2 cross-split overlaps   (parts of the 12 collisions)
   │
   ▼
TEST
   │
   ▼
2 train/test overlaps    (no parts of the 12 collisions)

Important comment before start TF-IDF

    The step performed are:

    Raw text
      ↓
    normalize whitespace
      ↓
    strip

    No execution of the following corrections:
    ❌ remove stopwords
    ❌ stemming
    ❌ lemmatization
    ❌ remove punctuation
    ❌ remove "not"


    Here the complete pipeline:
    
                Yelp Polarity
                      │
             560k train / 38k test
                      │
                      ▼
                Basic cleaning
                      │
                      ▼
              Train / Validation
                448k / 112k
                      │
                      ▼
                ┌───────────┐
                │  TF-IDF   │
                └─────┬─────┘
                      │
                      ▼
             Logistic Regression
                      │
                      ▼
          Validation evaluation
                      │
                      ▼
               Final test set